# 01 - Data Collection

This notebook runs the Tiki crawlers from `src/crawl/` to collect:
1. **Categories** – main and sub-categories from Tiki homepage
2. **Products** – products, prices, sellers, and reviews for selected categories (stored in PostgreSQL)

Prerequisites:
- `.env` configured (DB_*, SUPABASE_* if needed)
- Chrome installed (for Selenium)
- PostgreSQL running and migrated with `database/schema.sql`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "notebooks").exists():
    ROOT = ROOT 
else:
    ROOT = ROOT.parent if (ROOT.parent / "notebooks").exists() else ROOT

for p in [str(ROOT), str(ROOT / "src" / "crawl")]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Project root:", ROOT)

In [ ]:
from src.crawl import (
    create_all_tables,
    get_db_connection,
    run_category_crawler,
    run_product_crawler,
    setup_chrome_driver,
    TIKI_URL,
)

## 1. Ensure DB schema exists

Creates tables (category, seller, product, price_offer, reviewer, review) if they do not exist.

In [ ]:
conn = get_db_connection()
cur = conn.cursor()
create_all_tables(cur)
cur.close()
conn.close()
print("Schema ready.")

## 2. Crawl categories (optional)

Runs the category crawler: opens Tiki homepage, scrapes main and sub-categories, saves to DB.  
Then marks leaf categories for product crawl. **Run this first if you have no categories yet.**

In [ ]:
run_category_crawler()

## 3. Crawl products (and reviews)

Reads leaf categories with `is_scanned = TRUE` from DB (filter in `product_crawler.main`), then for each category:
- Loads product listing (with "See more" until done),
- For each product: detail page, seller, price/offer, reviews.

Data is written to PostgreSQL. To limit scope, edit the category filter in `src/crawl/product_crawler.py` (e.g. `Nhà Sách Tiki%`).

In [ ]:
run_product_crawler()

## 4. Check collected data (optional)

Quick row counts per table to verify the crawl.

In [ ]:
conn = get_db_connection()
cur = conn.cursor()
for table in ["category", "seller", "product", "price_offer", "reviewer", "review"]:
    cur.execute(f'SELECT COUNT(*) FROM "{table}"')
    n = cur.fetchone()[0]
    print(f"  {table}: {n}")
cur.close()
conn.close()